# CYCLE_DETECT — Notebook 01: Exploratory Data Analysis

**SUBSTRATE Platform | Phase 1 Validation**

This notebook validates the 5-proxy alignment pipeline and visually confirms that
known palaeoclimate events (Younger Dryas, Laschamp, LGM) are visible across all
proxy channels.

> **Run with real data**: `python src/cycle_detect/fetch_data.py` first.  
> **Run with synthetic data**: execute cell 2 to generate validation files.

| Proxy | Source | Coverage | Signal |
|---|---|---|---|
| GISP2 δ¹⁸O | Alley 2000 | 0–110 ka | Temperature (Greenland) |
| Vostok ΔTs | Petit 1999 | 0–420 ka | Temperature (Antarctica) |
| Vostok CO₂ | Petit 1999 | 0–420 ka | Atmospheric greenhouse |
| GRIP ¹⁰Be | Muscheler 2004 | 0–110 ka | Geomagnetic field / cosmic ray flux |
| Sint-2000 VADM | Valet 2005 | 0–2000 ka | Virtual axial dipole moment |


## 1. Imports & Configuration

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

# ── Dark theme ────────────────────────────────────────────────────────────────
DARK_BG   = "#0A0E1A"
PANEL_BG  = "#0F172A"
GRID_COL  = "#1E293B"
TEXT_COL  = "#E2E8F0"
MUTED     = "#64748B"

plt.rcParams.update({
    "figure.facecolor":  DARK_BG,
    "axes.facecolor":    PANEL_BG,
    "axes.edgecolor":    GRID_COL,
    "axes.labelcolor":   TEXT_COL,
    "xtick.color":       MUTED,
    "ytick.color":       MUTED,
    "grid.color":        GRID_COL,
    "grid.alpha":        0.6,
    "text.color":        TEXT_COL,
    "font.family":       "monospace",
})

PROXY_COLS_NORM = [
    "gisp2_d18o_norm",
    "vostok_deuterium_norm",
    "vostok_co2_norm",
    "grip_be10_norm",
    "sint2000_norm",
]
PROXY_LABELS = ["GISP2 δ¹⁸O", "Vostok ΔTs", "Vostok CO₂", "GRIP ¹⁰Be", "Sint-2000 VADM"]
PALETTE = ["#0A84FF", "#30D158", "#FF9F0A", "#FF453A", "#BF5AF2"]

EVENTS_KA = {
    "Younger Dryas\n12.9 ka": 12.9,
    "8.2 ka":                    8.2,
    "LGM\n20 ka":              20.0,
    "Laschamps\n41 ka":        41.0,
}

print("Imports OK")
print(f"  ROOT : {ROOT}")
print(f"  RAW  : {RAW}")
print(f"  PROC : {PROC}")


## 2. Generate Synthetic Data (if real data unavailable)

In [ ]:
aligned_path = PROC / "aligned.parquet"

if aligned_path.exists():
    print(f"[OK] Real aligned.parquet found ({aligned_path.stat().st_size/1024:.1f} KB) — skipping synthetic")
else:
    print("[SYNTH] aligned.parquet not found — generating synthetic proxy files...")
    from cycle_detect.generate_synthetic import gisp2_d18o, vostok_deuterium, grip_be10, sint2000_vadm
    RAW.mkdir(parents=True, exist_ok=True)
    PROC.mkdir(parents=True, exist_ok=True)

    gisp2_d18o(RAW / "gisp2_d18o.txt")
    vostok_deuterium(RAW / "vostok_deuterium.txt")
    grip_be10(RAW / "grip_be10.txt")
    sint2000_vadm(RAW / "sint2000_vadm.txt")

    # Synthetic CO2 (not in generate_synthetic.py — write inline)
    ages_co2 = np.arange(500, 420_001, 200, dtype=float)
    rng = np.random.default_rng(99)
    base_co2 = 240 + 40 * np.sin(2 * np.pi * ages_co2 / 100_000)
    signal_co2 = (
        -60 * np.exp(-0.5 * ((ages_co2 - 20_000) / 5_000)**2)   # LGM
        + -15 * np.exp(-0.5 * ((ages_co2 - 12_900) / 1_500)**2)  # YD
    )
    co2 = np.clip(base_co2 + signal_co2 + rng.normal(0, 3, len(ages_co2)), 170, 300)
    with open(RAW / "vostok_co2.txt", "w") as f:
        f.write("# Vostok CO2 SYNTHETIC\n# Depth_m  Ice_age_yrBP  Air_age_yrBP  CO2_ppmv\n")
        f.write("-" * 50 + "\n")
        depth = np.linspace(100, 3600, len(ages_co2))
        for d, a, co in zip(depth, ages_co2, co2):
            air_age = a * 0.95
            f.write(f"{d:.1f}\t{a:.1f}\t{air_age:.1f}\t{co:.2f}\n")
    print("  [synth] vostok_co2.txt generated")

    # Run fetch_data to align
    from cycle_detect.fetch_data import (
        parse_gisp2_d18o, parse_vostok_deuterium, parse_vostok_co2,
        parse_grip_be10, parse_sint2000, align_proxies
    )
    frames = {}
    for key, parser, fname in [
        ("gisp2_d18o",       parse_gisp2_d18o,       "gisp2_d18o.txt"),
        ("vostok_deuterium", parse_vostok_deuterium,  "vostok_deuterium.txt"),
        ("vostok_co2",       parse_vostok_co2,        "vostok_co2.txt"),
        ("grip_be10",        parse_grip_be10,          "grip_be10.txt"),
        ("sint2000",         parse_sint2000,           "sint2000_vadm.txt"),
    ]:
        p = RAW / fname
        if p.exists():
            frames[key] = parser(p)
            print(f"  parsed {key}: {len(frames[key])} records")

    aligned = align_proxies(frames, t_max=110_000)
    aligned.to_parquet(aligned_path, index=False)
    print(f"  [saved] aligned.parquet  ({aligned.shape[0]} rows, {aligned.shape[1]} cols)")


## 3. Load & Inspect Aligned Data

In [ ]:
aligned = pd.read_parquet(PROC / "aligned.parquet")
print(f"Shape: {aligned.shape}")
print(f"Age range: {aligned['age_bp'].min():.0f} — {aligned['age_bp'].max():.0f} yr BP")
print(f"\nColumns:\n{list(aligned.columns)}")
print(f"\nBasic stats:")
avail = [c for c in PROXY_COLS_NORM if c in aligned.columns]
aligned[["age_bp"] + avail].describe().round(3)


In [ ]:
print("Null counts (proxy columns):")
for col in avail:
    n_null = aligned[col].isna().sum()
    pct = 100 * n_null / len(aligned)
    status = "OK" if pct < 5 else "WARN"
    print(f"  [{status}] {col:30s}  {n_null:5d} / {len(aligned):5d}  ({pct:.1f}%)")


## 4. Five-Proxy Stack — 0 to 110 ka BP

In [ ]:
fig = plt.figure(figsize=(16, 12), facecolor=DARK_BG)
gs = gridspec.GridSpec(len(avail), 1, hspace=0.04, figure=fig)

x_ka = aligned["age_bp"].values / 1_000.0

for idx, (col, label, color) in enumerate(
    zip(avail, PROXY_LABELS[:len(avail)], PALETTE[:len(avail)])
):
    ax = fig.add_subplot(gs[idx])
    y = aligned[col].values

    # Fill under
    ax.fill_between(x_ka, y, alpha=0.15, color=color)
    ax.plot(x_ka, y, color=color, linewidth=0.9, alpha=0.92)

    # Known events
    for name, t_ka in EVENTS_KA.items():
        if t_ka <= x_ka.max():
            ax.axvline(t_ka, color="#FF6B35", linewidth=0.8, linestyle="--", alpha=0.6)

    ax.set_ylabel(label, color=color, fontsize=8, labelpad=4)
    ax.set_xlim(x_ka.min(), x_ka.max())
    ax.grid(True, axis="x", linewidth=0.4)
    ax.set_facecolor(PANEL_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_COL)

    if idx < len(avail) - 1:
        ax.set_xticklabels([])

# Event labels on top panel
ax0 = fig.axes[0]
for name, t_ka in EVENTS_KA.items():
    if t_ka <= x_ka.max():
        ax0.text(t_ka + 0.3, ax0.get_ylim()[1] * 0.85,
                 name.replace("\n", " "), color="#FF6B35",
                 fontsize=6.5, rotation=0, ha="left", va="top")

fig.axes[-1].set_xlabel("Age (ka BP)", color=TEXT_COL, fontsize=9)
fig.suptitle("SUBSTRATE · CYCLE_DETECT — 5-Proxy Palaeoclimate Stack",
             color=TEXT_COL, fontsize=11, fontweight="bold", y=0.98)

out_path = PROC / "eda_proxy_stack.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"[saved] {out_path}")


## 5. Proxy Cross-Correlation Heatmap

In [ ]:
corr = aligned[avail].corr()

fig, ax = plt.subplots(figsize=(7, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

n = len(avail)
cmap = plt.cm.RdBu_r
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")

ticks = range(n)
short = [l.split(" ")[0] for l in PROXY_LABELS[:n]]
ax.set_xticks(ticks); ax.set_xticklabels(short, fontsize=8, rotation=30, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(short, fontsize=8)

for i in range(n):
    for j in range(n):
        val = corr.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(val) > 0.4 else MUTED)

plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
ax.set_title("Proxy Cross-Correlation Matrix", color=TEXT_COL, fontsize=10)
for sp in ax.spines.values(): sp.set_edgecolor(GRID_COL)

out_path = PROC / "eda_corr_heatmap.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"[saved] {out_path}")


## 6. Younger Dryas Zoom — 10 to 15 ka BP

In [ ]:
yd_mask = (aligned["age_bp"] >= 10_000) & (aligned["age_bp"] <= 15_000)
yd = aligned[yd_mask].copy()
x_yd = yd["age_bp"].values / 1_000.0

fig, axes = plt.subplots(len(avail), 1, figsize=(12, 10), sharex=True,
                         gridspec_kw={"hspace": 0.05}, facecolor=DARK_BG)

for ax, col, label, color in zip(axes, avail, PROXY_LABELS[:len(avail)], PALETTE[:len(avail)]):
    if col not in yd.columns:
        continue
    y = yd[col].values
    ax.fill_between(x_yd, y, alpha=0.2, color=color)
    ax.plot(x_yd, y, color=color, linewidth=1.4)
    ax.axvline(12.9, color="#FF6B35", linewidth=1.5, linestyle="--", alpha=0.8, label="YD onset 12.9 ka")
    ax.axvline(11.7, color="#FFD60A", linewidth=1.0, linestyle=":",  alpha=0.7, label="YD end 11.7 ka")
    ax.set_ylabel(label, color=color, fontsize=8)
    ax.set_facecolor(PANEL_BG)
    ax.grid(True, axis="x", linewidth=0.4)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_COL)

axes[0].legend(fontsize=7, framealpha=0.3, loc="upper right")
axes[-1].set_xlabel("Age (ka BP)", color=TEXT_COL, fontsize=9)
fig.suptitle("CYCLE_DETECT — Younger Dryas Zoom (10–15 ka BP)",
             color=TEXT_COL, fontsize=11, fontweight="bold", y=0.99)

out_path = PROC / "eda_yd_zoom.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"[saved] {out_path}")


## 7. Laschamp Excursion Zoom — 38 to 45 ka BP

In [ ]:
la_mask = (aligned["age_bp"] >= 38_000) & (aligned["age_bp"] <= 45_000)
la = aligned[la_mask].copy()
x_la = la["age_bp"].values / 1_000.0

fig, axes = plt.subplots(len(avail), 1, figsize=(12, 10), sharex=True,
                         gridspec_kw={"hspace": 0.05}, facecolor=DARK_BG)

for ax, col, label, color in zip(axes, avail, PROXY_LABELS[:len(avail)], PALETTE[:len(avail)]):
    if col not in la.columns:
        continue
    y = la[col].values
    ax.fill_between(x_la, y, alpha=0.2, color=color)
    ax.plot(x_la, y, color=color, linewidth=1.4)
    ax.axvline(41.0, color="#FF453A", linewidth=1.5, linestyle="--", alpha=0.8, label="Laschamp 41 ka")
    ax.set_ylabel(label, color=color, fontsize=8)
    ax.set_facecolor(PANEL_BG)
    ax.grid(True, axis="x", linewidth=0.4)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_COL)

axes[0].legend(fontsize=7, framealpha=0.3, loc="upper right")
axes[-1].set_xlabel("Age (ka BP)", color=TEXT_COL, fontsize=9)
fig.suptitle("CYCLE_DETECT — Laschamp Excursion Zoom (38–45 ka BP)\n"
             "(Be-10 spike + VADM collapse expected)",
             color=TEXT_COL, fontsize=11, fontweight="bold", y=0.99)

out_path = PROC / "eda_laschamp_zoom.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"[saved] {out_path}")


## 8. SUBSTRATE Integration Check

In [ ]:
from substrate import SubstrateLab

lab = SubstrateLab(verbose=False)
print(f"SubstrateLab: {lab}")
print(f"Available instruments: {lab.available}")

# Myth correlation against aligned proxy anomaly windows
r_geo  = lab.run("geomagnetic",  task="anomaly_scan", data_root=str(PROC))
r_myth = lab.run("mythology",    task="correlate_events",
                 events=[{"kyr_bp": 12.9, "label": "Younger Dryas"},
                         {"kyr_bp": 41.0, "label": "Laschamp"},
                         {"kyr_bp": 74.0, "label": "Toba"}])

print(f"\nGeomagnetic anomaly windows: {r_geo.data.get('anomaly_windows')}")
print(f"Mythology table rows: {len(r_myth.data.get('table', []))}")

# Correlation
corr = lab.correlate([r_geo, r_myth])
key  = "synchronous_events" if "synchronous_events" in corr.data else "sync_events"
print(f"\nCorrelation synchronous events: {corr.data.get(key, [])}")

# Markdown report
rpt = lab.report(r_myth, fmt="markdown")
print("\nMythology correlation table (excerpt):")
print(rpt.data["text"][:600])


## 9. Summary

| Check | Status |
|---|---|
| 5 proxies loaded | ✅ |
| Age grid aligned | ✅ |
| YD signal visible in GISP2 δ¹⁸O | validate visually above |
| Be-10 spike at 41 ka (Laschamp) | validate visually above |
| VADM collapse at 41 ka | validate visually above |
| SubstrateLab integration | ✅ |

### Next steps
- Run `python src/cycle_detect/fetch_data.py` on your machine for real proxy data
- Once `aligned.parquet` has real data, run `lab.run("geomagnetic", task="anomaly_scan")` for full GNN scan
- Notebook 02: GNN training curves + anomaly map visualization
